# Phase 4 — Smoke Test

Verify the MIL pipeline works end-to-end on one fold (All streams, phase-resolved
attention, test=2020) before launching the full grid.

Should take ~3–8 minutes on the MIG A100 slice.

What to check in the output:
1. **Loss decreases monotonically** across the 15 epochs.
2. **Memory stays comfortably under 5 GB** GPU (model is small).
3. **Station-level macro MAE is in the ballpark of 6–8 days** (similar to Phase 3 LSTM).
4. **Attention weights vary across phases** (alpha shape is (N_bags, 10, 7); we want
   to see different weights for different phases — that's the headline result).


## 1. Setup

In [1]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB)')

def memstat(tag=''):
    cpu = psutil.Process(os.getpid()).memory_info().rss / 1e9
    if torch.cuda.is_available():
        return f'{tag} RAM={cpu:.2f}GB  GPU_alloc={torch.cuda.memory_allocated()/1e9:.2f}GB'
    return f'{tag} RAM={cpu:.2f}GB'

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
print(memstat('init:'))


Device: cuda
GPU: NVIDIA A100-PCIE-40GB MIG 1g.5gb  (5.10 GB)
init: RAM=0.40GB  GPU_alloc=0.00GB


## 2. Load one fold (4 train years + 2020 test) and build bags

In [2]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase4_mil.bag_dataset import build_bags

YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2020

# We use the all10 policy: include all 10 fields per station (no filtering)
policy = SelectionPolicy(use_only_rank1=False)

print('Loading test set...')
test_b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{TEST_YEAR}.npz'), policy)
test_bags = build_bags(test_b)
print(f'  test: {len(test_b["xid"])} fields → {test_bags["X"].shape[0]} bags')

print('Loading train set...')
parts = {}
for y in YEARS:
    if y == TEST_YEAR: continue
    parts[y] = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
train_b = concat_bundles(parts, years=[y for y in YEARS if y != TEST_YEAR])
del parts; gc.collect()
train_bags = build_bags(train_b)
del train_b; gc.collect()
print(f'  train bags: {train_bags["X"].shape}')
print(memstat('after load:'))


Loading test set...
  test: 5181 fields → 585 bags
Loading train set...
  train bags: (2424, 10, 426, 19)
after load: RAM=1.94GB  GPU_alloc=0.00GB


## 3. Run one fold (All streams, phase-resolved attention)

In [3]:
from phase4_mil.train import train_mil_fold, STREAM_CONFIGS

t0 = time.time()
result = train_mil_fold(
    train_bags=train_bags, test_bags=test_bags,
    active=STREAM_CONFIGS['All'],
    hidden=32, epochs=15, batch_size=8, lr=1e-3,
    phase_resolved_attention=True, field_loss_weight=0.3,
    use_amp=True, device=DEVICE, verbose=True,
)
print(f'\nFold completed in {time.time()-t0:.1f}s')
print(memstat('after train:'))


    epoch   1: total=0.1531  st=0.1205  fd=0.1087
    epoch   2: total=0.0839  st=0.0643  fd=0.0653
    epoch   3: total=0.0813  st=0.0622  fd=0.0637
    epoch   4: total=0.0805  st=0.0616  fd=0.0631
    epoch   5: total=0.0800  st=0.0612  fd=0.0627
    epoch   6: total=0.0796  st=0.0609  fd=0.0623
    epoch   7: total=0.0792  st=0.0606  fd=0.0620
    epoch   8: total=0.0789  st=0.0604  fd=0.0617
    epoch   9: total=0.0789  st=0.0604  fd=0.0618
    epoch  10: total=0.0786  st=0.0602  fd=0.0616
    epoch  11: total=0.0788  st=0.0602  fd=0.0618
    epoch  12: total=0.0784  st=0.0600  fd=0.0615
    epoch  13: total=0.0784  st=0.0599  fd=0.0614
    epoch  14: total=0.0782  st=0.0598  fd=0.0612
    epoch  15: total=0.0782  st=0.0598  fd=0.0613

Fold completed in 162.5s
after train: RAM=2.52GB  GPU_alloc=0.02GB


## 4. Evaluate both levels

In [4]:
from phase3_baselines.evaluate import evaluate_predictions, print_evaluation

phase_ids = train_bags['phase_ids']

# Station-level eval
print('==== STATION-LEVEL EVALUATION ====')
report_station = evaluate_predictions(
    pred_days=result['station_pred'].astype(np.float32),
    true_days=result['station_truth'].astype(np.float32),
    phase_ids=phase_ids,
    years=result['station_year'],
    ranks=None,
)
print_evaluation(report_station, 'Station-level (one prediction per station)')


==== STATION-LEVEL EVALUATION ====

Station-level (one prediction per station)

[Headline — macro over phases]
  macro_mae               : 8.218
  macro_r2                : -0.018
  macro_within_5d         : 0.436
  n_phases_evaluated      : 7

[Per phase]
         n     mae    rmse     r2  within_5d   bias
phase                                              
10     528  10.241  12.905 -0.128      0.333 -4.271
12     518  10.324  13.171  0.031      0.361 -2.931
15     489   9.344  11.721 -0.195      0.380  3.965
18     537   8.527  10.497 -0.555      0.363  7.019
19     471   7.580   9.872  0.115      0.463  0.573
21     500   6.636   8.696  0.250      0.518  1.268
24     563   4.877   6.644  0.353      0.636 -1.293

[Per (phase, year)]
year    2020
phase       
10     10.24
12     10.32
15      9.34
18      8.53
19      7.58
21      6.64
24      4.88


In [5]:
# Field-level eval
print('==== FIELD-LEVEL EVALUATION ====')
report_field = evaluate_predictions(
    pred_days=result['field_pred'].astype(np.float32),
    true_days=result['field_truth'].astype(np.float32),
    phase_ids=phase_ids,
    years=result['field_year'],
    ranks=result['field_rank'],
)
print_evaluation(report_field, 'Field-level (one prediction per field)')


==== FIELD-LEVEL EVALUATION ====

Field-level (one prediction per field)

[Headline — macro over phases]
  macro_mae               : 8.684
  macro_r2                : -0.153
  macro_within_5d         : 0.422
  n_phases_evaluated      : 7

[Per phase]
          n     mae    rmse     r2  within_5d   bias
phase                                               
10     4719  11.984  14.740 -0.459      0.273 -7.655
12     4628  10.449  13.219  0.006      0.352 -2.972
15     4321  10.898  13.919 -0.723      0.300  6.389
18     4758   6.875   8.959 -0.199      0.501  3.735
19     4170   7.912  10.258  0.025      0.454 -0.371
21     4422   7.100   9.278  0.132      0.491 -0.255
24     4986   5.569   7.427  0.149      0.584 -0.598

[Per (phase, year)]
year    2020
phase       
10     11.98
12     10.45
15     10.90
18      6.87
19      7.91
21      7.10
24      5.57

[Per (phase, rank) — MAE]
rank      1      2      3      4      5      6      7      8      9      10
phase                          

## 5. Inspect attention weights

In [6]:
# alpha shape: (N_bags, 10, P)
alpha = result['alpha']
print(f'alpha shape: {alpha.shape}')

# Average attention weight per (rank, phase)
# Each bag has fields sorted by rank in slots 0–9 (rank 1 at slot 0)
print('\n=== Mean attention weight by (slot, phase) ===')
mean_alpha = alpha.mean(axis=0)   # (10, P)
df = pd.DataFrame(mean_alpha.round(3),
                    index=[f'slot{i+1}_rank≤{i+1}' for i in range(10)],
                    columns=[f'phase_{p}' for p in phase_ids])
print(df.to_string())

# Are weights phase-discriminative? Compare variance across slots for each phase
print('\n=== Coefficient of variation across slots, per phase (higher = more selective) ===')
cv = mean_alpha.std(axis=0) / (mean_alpha.mean(axis=0) + 1e-8)
for p, c in zip(phase_ids, cv):
    print(f'  phase {p}: CV = {c:.3f}')


alpha shape: (585, 10, 7)

=== Mean attention weight by (slot, phase) ===
                phase_10  phase_12  phase_15  phase_18  phase_19  phase_21  phase_24
slot1_rank≤1       0.142     0.142     0.146     0.146     0.146     0.146     0.147
slot2_rank≤2       0.122     0.122     0.125     0.126     0.125     0.125     0.126
slot3_rank≤3       0.116     0.116     0.113     0.113     0.113     0.113     0.112
slot4_rank≤4       0.103     0.103     0.103     0.103     0.103     0.103     0.103
slot5_rank≤5       0.096     0.096     0.095     0.095     0.095     0.095     0.095
slot6_rank≤6       0.091     0.091     0.091     0.091     0.091     0.091     0.092
slot7_rank≤7       0.088     0.087     0.086     0.086     0.086     0.086     0.086
slot8_rank≤8       0.085     0.084     0.083     0.083     0.083     0.083     0.083
slot9_rank≤9       0.081     0.081     0.080     0.080     0.080     0.080     0.080
slot10_rank≤10     0.078     0.078     0.076     0.076     0.076     0.076  

## 6. Interpretation guide

Look for:

- **Station-level macro_mae** in 6–8 day range. If it's worse than Phase 3 LSTM (8 days for All/all10),
  the MIL formulation is hurting and we need to diagnose.
- **Field-level macro_mae** should be similar to or slightly worse than Phase 3 LSTM all10
  (which was 8.07 for All). The interesting comparison is station vs field within MIL itself.
- **Attention CV per phase**: if CV is 0.0 for every phase, attention is uniform and
  effectively doing mean pooling — no value-add from MIL. If CV varies meaningfully
  across phases (e.g., 0.1 for phase 18, 0.5 for phase 24), MIL is doing real work.

If everything looks good, proceed to `05_mil_full.ipynb`.
